In [2]:
import pandas as pd
import re
import zstandard as zstd
import io
import orjson
from collections import defaultdict
import json
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import matplotlib.pyplot as plt
import csv
import polars as pl
import os
import gdown

In [34]:
datasets = {
    "2024-01": {
        "comments": "1uvGUsCub1MO0P2m3KfDN02e7qJRxGtNN",
        "submissions": "1d5lIEcaVXmWrWFtxBwf6cNzl8IqSSonO"},
    "2024-02": {
        "comments": "14DDZSevjpCKLt72xf9xCieehGA3_721r", 
        "submissions": "1-ysVulah19XW-kiphPQ6y_rh1xvTp7-b"},
    "2024-03": {
        "comments": "1u-V2SwM8SvNZICouLYxGYomdrCy7NMKQ",
        "submissions": "1QTW0Zjn2UuxoHzglMU9ayZIq3eBDyTWp"},
    "2024-04": {
        "comments": "1tMzO5j9CTMEPu3uLoKAZryiNStGg03q8",
        "submissions": "1fBd_RTLkaf0xGM6LOXMubsvo-gQ7XAYI"},
    "2024-05": {
        "comments": "11_67msmdxrKiMTkUOWVRbKTXtCJMU8HR",
        "submissions": "1pUq9hA6o7cbRh3pYrEaUuMgcufgLSy-V"},
    "2024-06": {
        "comments": "1v_DeMNDISk6A8rnUobbT3WoHtOQAbd1H",
        "submissions": "1FzfUGzG6d1wQ-eR1rExDBpDIZicEPb_O"}
}

In [35]:
data_dir = "data"

def load_reddit_data(month: str, data_type: str):
    """
    month: '2024-03'
    data_type: 'comments' or 'submissions'
    """
    os.makedirs(data_dir, exist_ok=True)

    file_id = datasets[month][data_type]
    filename = f"{data_type}_{month}.parquet"
    path = os.path.join(data_dir, filename)

    if not os.path.exists(path):
        print(f"Downloading {file_id} from Google Drive")
        gdown.download(id=file_id, output=path, quiet=False)

    return pl.read_parquet(path)

In [36]:
# download and load the datasets

for month, type_dict in datasets.items():
    for data_type, file_id in type_dict.items(): 
        if file_id == "File_id":
            continue
        else: 
            df_subs = load_reddit_data(month, data_type)

Downloading...
From (original): https://drive.google.com/uc?id=14DDZSevjpCKLt72xf9xCieehGA3_721r
From (redirected): https://drive.google.com/uc?id=14DDZSevjpCKLt72xf9xCieehGA3_721r&confirm=t&uuid=1a4e4868-c8a4-4fcd-a099-2e8443e93d62
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/comments_2024-02.parquet
100%|████████████████████████████████████████| 495M/495M [02:43<00:00, 3.02MB/s]


Downloading...
From: https://drive.google.com/uc?id=1-ysVulah19XW-kiphPQ6y_rh1xvTp7-b
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/submissions_2024-02.parquet
100%|██████████████████████████████████████| 30.7M/30.7M [00:16<00:00, 1.90MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1tMzO5j9CTMEPu3uLoKAZryiNStGg03q8
From (redirected): https://drive.google.com/uc?id=1tMzO5j9CTMEPu3uLoKAZryiNStGg03q8&confirm=t&uuid=338133dd-1c44-4ba3-b5d5-9a7102971890
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/comments_2024-04.parquet
100%|████████████████████████████████████████| 513M/513M [02:37<00:00, 3.25MB/s]


Downloading...
From: https://drive.google.com/uc?id=1fBd_RTLkaf0xGM6LOXMubsvo-gQ7XAYI
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/submissions_2024-04.parquet
100%|██████████████████████████████████████| 31.2M/31.2M [00:07<00:00, 4.21MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=11_67msmdxrKiMTkUOWVRbKTXtCJMU8HR
From (redirected): https://drive.google.com/uc?id=11_67msmdxrKiMTkUOWVRbKTXtCJMU8HR&confirm=t&uuid=cfffd576-f2ef-4b88-8ed0-d99efe336361
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/comments_2024-05.parquet
100%|████████████████████████████████████████| 385M/385M [01:36<00:00, 4.01MB/s]


Downloading...
From: https://drive.google.com/uc?id=1pUq9hA6o7cbRh3pYrEaUuMgcufgLSy-V
To: /Users/donylalcantara/Documents/AIM/Final_ACCESS/Accesslab_MiniProject_Alcantara-Vocal/data/submissions_2024-05.parquet
100%|██████████████████████████████████████| 30.6M/30.6M [00:07<00:00, 4.20MB/s]


In [9]:
march_test = pl.scan_parquet('parquet/comments_2024_march_full.parquet')
march_test.collect().head(10)

author,subreddit,count_comments
str,str,i64
"""mouseat9""","""aitah""",1
"""LargeBarnacle7711""","""dnd""",1
"""sgh616""","""foodporn""",1
"""Onlyroad4adrifter""","""facepalm""",1
"""ewejoser""","""worldnews""",1
"""guest_username2""","""peterexplainsthejoke""",1
"""learnyouathang""","""askreddit""",1
"""HonestBalloon""","""politics""",1
"""Synyster182""","""unpopularopinion""",1


In [27]:
# load march comments into polar df

df_mc = pl.scan_parquet("data/comments_2024-03.parquet")
df_mc.collect()

author,subreddit,count_comments
str,str,i64
"""mouseat9""","""aitah""",1
"""LargeBarnacle7711""","""dnd""",1
"""sgh616""","""foodporn""",1
"""Onlyroad4adrifter""","""facepalm""",1
"""ewejoser""","""worldnews""",1
…,…,…
"""Prowindowlicker""","""atheism""",1
"""Time-Ad-7055""","""facepalm""",1
"""wholeearthmama""","""aww""",1


In [28]:
# aggregate march comments

df_mc_agg = df_mc.group_by(['author', 'subreddit']).agg(
    pl.col("count_comments").sum().alias('count_comments')).sort('author')

df_mc_agg.collect()

author,subreddit,count_comments
str,str,i64
"""------------------16""","""characterai""",5
"""------------------GL""","""facepalm""",2
"""------------------GL""","""mildlyinteresting""",9
"""------------------GL""","""beamazed""",1
"""------------------GL""","""gaming""",1
…,…,…
"""zzzzzzzzzzzzvzzzzvzz""","""startups""",3
"""zzzzzzzzzzzzzzzz0""","""trueoffmychest""",2
"""zzzzzzzzzzzzzzzz0""","""college""",9


In [31]:
# load march sub into polar df
df_ms = pl.scan_parquet("data/submissions_2024-03.parquet")
df_ms.collect()

author,subreddit,count_posts
str,str,i64
"""IntroductionFit4364""","""coffee""",1
"""mattthheww""","""indieheads""",1
"""3kOlen""","""popheads""",1
"""Key-Tie1861""","""teenagers""",1
"""WillamThunderfuck""","""popheads""",1
…,…,…
"""barki26""","""finance""",1
"""Ashell77""","""askreddit""",1
"""Live-Ad-6279""","""houseofthedragon""",1


In [32]:
# aggregate march submissions

df_ms_agg = df_ms.group_by(['author', 'subreddit']).agg(
    pl.col("count_posts").sum().alias('count_posts')).sort('author')

df_ms_agg.collect()

author,subreddit,count_posts
str,str,i64
"""---------00---------""","""unexpected""",1
"""---------00---------""","""askreddit""",3
"""--------51""","""youtube""",1
"""-------7654321""","""nostupidquestions""",1
"""-------Tom---------""","""askreddit""",1
…,…,…
"""zzzzzzzzzra""","""cooking""",3
"""zzzzzzzzzra""","""books""",1
"""zzzzzzzzzra""","""astronomy""",1


In [33]:
# merge aggregated march comments and posts

# merge aggregated march submissions and comments 
march_merged = df_mc_agg.join(
    df_ms_agg,
    on=["author", "subreddit"],
    how="full",
    coalesce=True
).sort('author')

# fill nans for count comments and posts
march_merged = march_merged.with_columns(pl.col('count_comments').fill_null(strategy="zero"),
                                         pl.col('count_posts').fill_null(strategy="zero"))

# show total interaction for march (comment + sub)
march_merged = march_merged.with_columns((pl.col('count_comments') + pl.col('count_posts')).alias('total_interactions'))

# show
march_merged.collect()

author,subreddit,count_comments,count_posts,total_interactions
str,str,i64,i64,i64
"""------------------16""","""characterai""",5,0,5
"""------------------GL""","""personalfinance""",2,0,2
"""------------------GL""","""videos""",1,0,1
"""------------------GL""","""howto""",9,0,9
"""------------------GL""","""diy""",11,0,11
…,…,…,…,…
"""zzzzzzzzzzzzvzzzzvzz""","""startups""",3,0,3
"""zzzzzzzzzzzzzzzz0""","""trueoffmychest""",2,0,2
"""zzzzzzzzzzzzzzzz0""","""dating""",5,0,5
